In [1]:
import torch
import numpy as np
from src.data.sequence import Sequence  # 替换为你的实际模块路径

# 设置新的时间范围和事件数量
t_start = 0.0
t_end = 20.0  # 设置更长的结束时间
num_events = 10  # 新的事件数量

# 随机生成事件发生的时间
arrival_times = np.sort(np.random.uniform(t_start, t_end, num_events))  # 随机生成 10 个事件的到达时间
print("Generated arrival times:", arrival_times)

# 计算 inter_times: Δt_0, Δt_1, ..., Δt_N, Δt_survival
inter_times = np.diff(arrival_times, prepend=[t_start], append=[t_end])  # 计算事件之间的时间差

# 随机生成震级和位置
mag = np.random.uniform(2.5, 5.0, num_events)  # 随机生成震级
loc = np.random.uniform(30.0, 31.0, (num_events, 2))  # 随机生成每个事件的经纬度位置

# 创建 Sequence 对象
seq = Sequence(
    inter_times=inter_times,
    t_start=t_start,
    mag=mag,
    loc=loc
)

# ✅ 查看生成的完整事件序列
print("Original arrival times:", seq.arrival_times)
print("Original inter_times:", seq.inter_times)
print("Original magnitude:", seq['mag'])

# ✅ 调用 get_subsequence（例如截取 [5.0, 15.0] 之间的事件）
sub_seq = seq.get_subsequence(start=5.0, end=15.0)

# ✅ 输出子序列内容
print("\n--- Subsequence ---")
print("Arrival times:", sub_seq.arrival_times)
print("Inter times:", sub_seq.inter_times)
print("Magnitude:", sub_seq['mag'])
print("Location:", sub_seq['loc'])


Generated arrival times: [ 1.98395229  5.80654034  6.37316736  7.08248604  7.09276607  7.99174348
 13.63770529 15.18179832 16.97218097 19.46947812]
Original arrival times: tensor([ 1.9840,  5.8065,  6.3732,  7.0825,  7.0928,  7.9917, 13.6377, 15.1818,
        16.9722, 19.4695], dtype=torch.float64)
Original inter_times: tensor([1.9840, 3.8226, 0.5666, 0.7093, 0.0103, 0.8990, 5.6460, 1.5441, 1.7904,
        2.4973, 0.5305], dtype=torch.float64)
Original magnitude: tensor([3.4304, 3.2373, 3.9067, 3.3437, 2.5428, 3.4167, 3.3753, 3.8913, 3.1881,
        3.5302], dtype=torch.float64)

--- Subsequence ---
Arrival times: tensor([ 5.8065,  6.3732,  7.0825,  7.0928,  7.9917, 13.6377],
       dtype=torch.float64)
Inter times: tensor([0.8065, 0.5666, 0.7093, 0.0103, 0.8990, 5.6460, 1.3623],
       dtype=torch.float64)
Magnitude: tensor([3.2373, 3.9067, 3.3437, 2.5428, 3.4167, 3.3753], dtype=torch.float64)
Location: tensor([[30.2392, 30.1470],
        [30.1265, 30.1631],
        [30.9885, 30.1984]

In [2]:
sub_seq.inter_times.shape

torch.Size([7])

In [3]:
from src.data.batch import Batch 
buffer_batch = Batch.init_sample_batch(past_seq=sub_seq, batch_size=2, max_sample_len=5)

In [4]:
print(buffer_batch)

Batch(
  inter_times: [2, 12],
  arrival_times: [2, 12],
  t_start: [2],
  t_end: [2],
  t_nll_start: [2],
  nll_event_mask: [2, 12],
  input_mask: [2, 12],
  start_idx: [2],
  end_idx: [2],
  non_pad_mask: [2, 12],
  type_seq: [2, 12],
  mag: [2, 12],
  loc: [2, 12, 2]
)


In [5]:
sample_batch = buffer_batch.get_sample_batch()

In [6]:
print(sample_batch)

Batch(
  inter_times: [2, 6],
  arrival_times: [2, 6],
  end_idx: [2],
  input_mask: [2, 6],
  nll_event_mask: [2, 6],
  non_pad_mask: [2, 6],
  type_seq: [2, 6],
  mag: [2, 6],
  loc: [2, 6, 2]
)


In [7]:
buffer_batch2 = Batch.init_sample_batch(past_seq=sub_seq.get_subsequence(sub_seq.t_end, sub_seq.t_end), batch_size=2, max_sample_len=5)

In [8]:
buffer_batch2.arrival_times

tensor([[15., 25., 35., 45., 55., 65.],
        [15., 25., 35., 45., 55., 65.]], dtype=torch.float64)

In [9]:
buffer_batch.t_start, buffer_batch.t_end

(tensor([5., 5.], dtype=torch.float64),
 tensor([15., 15.], dtype=torch.float64))

In [10]:
buffer_batch2.t_start, buffer_batch2.t_end

(tensor([15., 15.], dtype=torch.float64),
 tensor([15., 15.], dtype=torch.float64))

In [11]:
sampel_batch2 = buffer_batch2.get_sample_batch()

In [12]:
sample_batch.inter_times

tensor([[0.8065, 0.5666, 0.7093, 0.0103, 0.8990, 5.6460],
        [0.8065, 0.5666, 0.7093, 0.0103, 0.8990, 5.6460]], dtype=torch.float64)

In [13]:
sampel_batch2.inter_times

tensor([], size=(2, 0), dtype=torch.float64)

In [14]:
t =sample_batch.inter_times 
t= t.unsqueeze(-1)
print(t[:,-1:].shape)

torch.Size([2, 1, 1])


In [15]:
sub_event_seq = sub_seq.to_event_sequence()

In [37]:
sub_seq.inter_times.shape

torch.Size([7])

In [ ]:
sub_seq.arrival_times[-2]

In [33]:
sub_seq.init_sample_sequence().arrival_times

/root/autodl-tmp/chuandian_eq/src/data/sequence.py:188: UserWarning: Found 1 zero inter-event times in the sequence. This violates fundamental assumptions of TPP models and may lead to incorrect log-likelihood values.
  warnings.warn(


tensor([7.9917], dtype=torch.float64)

In [18]:
print(sub_event_seq.arrival_times,sub_event_seq.inter_times)

tensor([0.0000, 0.5666, 1.2759, 1.2862, 2.1852, 7.8312], dtype=torch.float64) tensor([0.0000, 0.5666, 0.7093, 0.0103, 0.8990, 5.6460], dtype=torch.float64)


In [19]:
print(sub_seq.arrival_times)

tensor([ 5.8065,  6.3732,  7.0825,  7.0928,  7.9917, 13.6377],
       dtype=torch.float64)


In [20]:
from src.data.tpp_dataset import TppDataset
ds = TppDataset([seq])

In [21]:
dl = ds.get_dataloader(batch_size=2, shuffle=True)

In [22]:
for batch in dl:
    print("\n--- Batch ---")
    print("Arrival times:", batch['arrival_times'])
    print("Inter times:", batch['inter_times'])
    print("Magnitude:", batch['mag'])
    print("Location:", batch['loc'])
    break  # 只打印第一个批次


--- Batch ---
Arrival times: tensor([[ 1.9840,  5.8065,  6.3732,  7.0825,  7.0928,  7.9917, 13.6377, 15.1818,
         16.9722, 19.4695, 20.0000]], dtype=torch.float64)
Inter times: tensor([[1.9840, 3.8226, 0.5666, 0.7093, 0.0103, 0.8990, 5.6460, 1.5441, 1.7904,
         2.4973, 0.5305]], dtype=torch.float64)
Magnitude: tensor([[ 3.4304,  3.2373,  3.9067,  3.3437,  2.5428,  3.4167,  3.3753,  3.8913,
          3.1881,  3.5302, 10.0000]], dtype=torch.float64)
Location: tensor([[[30.6900, 30.2168],
         [30.2392, 30.1470],
         [30.1265, 30.1631],
         [30.9885, 30.1984],
         [30.2625, 30.9644],
         [30.1829, 30.1247],
         [30.5037, 30.6986],
         [30.3485, 30.3399],
         [30.1781, 30.9078],
         [30.3237, 30.3595],
         [10.0000, 10.0000]]], dtype=torch.float64)


In [23]:
event_ds = TppDataset([sub_event_seq]) 
event_dl = event_ds.get_dataloader(batch_size=2, shuffle=True)
for batch in event_dl:
    print("\n--- Event Batch ---")
    print("Arrival times:", batch['arrival_times'])
    print("Inter times:", batch['inter_times'])
    print("Magnitude:", batch['mag'])
    print("Location:", batch['loc'])
    print("Type seq:", batch['type_seq'])
    break  # 只打印第一个批次


--- Event Batch ---
Arrival times: tensor([[0.0000, 0.5666, 1.2759, 1.2862, 2.1852, 7.8312]])
Inter times: tensor([[0.0000, 0.5666, 0.7093, 0.0103, 0.8990, 5.6460]])
Magnitude: tensor([[3.2373, 3.9067, 3.3437, 2.5428, 3.4167, 3.3753]], dtype=torch.float64)
Location: tensor([[[30.2392, 30.1470],
         [30.1265, 30.1631],
         [30.9885, 30.1984],
         [30.2625, 30.9644],
         [30.1829, 30.1247],
         [30.5037, 30.6986]]], dtype=torch.float64)
Type seq: tensor([[0, 0, 0, 0, 0, 0]])


In [24]:
import torch

def get_time_window_mask(timestamps: torch.Tensor, window_size: float) -> torch.Tensor:
    """
    Sliding-window causal mask  (True=被掩蔽, False=可见)
    """
    diff = timestamps.unsqueeze(2) - timestamps.unsqueeze(1)   
    mask = (diff < 0) | (diff > window_size)            
    return mask.bool()

# 两条序列各 6 个时间戳
t = torch.tensor([[0., 1., 3., 8., 12., 13.],
                  [0., 2., 4., 7., 11., 20.]])

mask = get_time_window_mask(t, window_size=5.)

print("序列 0 的掩码 (0=可见, 1=被掩蔽):")
print(mask[0].int())
print("\n序列 1 的掩码:")
print(mask[1].int())


序列 0 的掩码 (0=可见, 1=被掩蔽):
tensor([[0, 1, 1, 1, 1, 1],
        [0, 0, 1, 1, 1, 1],
        [0, 0, 0, 1, 1, 1],
        [1, 1, 0, 0, 1, 1],
        [1, 1, 1, 0, 0, 1],
        [1, 1, 1, 0, 0, 0]], dtype=torch.int32)

序列 1 的掩码:
tensor([[0, 1, 1, 1, 1, 1],
        [0, 0, 1, 1, 1, 1],
        [0, 0, 0, 1, 1, 1],
        [1, 0, 0, 0, 1, 1],
        [1, 1, 1, 0, 0, 1],
        [1, 1, 1, 1, 1, 0]], dtype=torch.int32)


In [25]:
def get_subsequent_mask(seq):
    """ For masking out the subsequent info, i.e., masked self-attention. """
    assert seq.dim() == 2
    sz_b, len_s = seq.size()
    subsequent_mask = torch.triu(
        torch.ones((len_s, len_s), device=seq.device, dtype=torch.uint8), diagonal=1)
    subsequent_mask = subsequent_mask.unsqueeze(0).expand(sz_b, -1, -1)  # b x ls x ls
    return subsequent_mask

get_subsequent_mask(t)

tensor([[[0, 1, 1, 1, 1, 1],
         [0, 0, 1, 1, 1, 1],
         [0, 0, 0, 1, 1, 1],
         [0, 0, 0, 0, 1, 1],
         [0, 0, 0, 0, 0, 1],
         [0, 0, 0, 0, 0, 0]],

        [[0, 1, 1, 1, 1, 1],
         [0, 0, 1, 1, 1, 1],
         [0, 0, 0, 1, 1, 1],
         [0, 0, 0, 0, 1, 1],
         [0, 0, 0, 0, 0, 1],
         [0, 0, 0, 0, 0, 0]]], dtype=torch.uint8)

In [26]:
import torch
buffer_batch2 = torch.load('buffer_batch2.pt')

/root/autodl-tmp/tmp/ipykernel_11795/2824022099.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  buffer_batch2 = torch.load('buffer_batch2.pt')


In [27]:
buffer_batch2

Batch(
  inter_times: [100, 2002],
  arrival_times: [100, 2002],
  t_start: [100],
  t_end: [100],
  t_nll_start: [100],
  nll_event_mask: [100, 2002],
  input_mask: [100, 2002],
  start_idx: [100],
  end_idx: [100],
  non_pad_mask: [100, 2002],
  type_seq: [100, 2002],
  mag: [100, 2002],
  loc: [100, 2002, 2],
  depth: [100, 2002]
)

In [28]:
buffer_batch2.get_sample_batch()

Batch(
  inter_times: [100, 1],
  arrival_times: [100, 1],
  end_idx: [100],
  input_mask: [100, 1],
  nll_event_mask: [100, 1],
  non_pad_mask: [100, 1],
  type_seq: [100, 1],
  mag: [100, 1],
  loc: [100, 1, 2],
  depth: [100, 1]
)

In [29]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inter_times = torch.randn(100,1).to(device)
mag = torch.randn(100,1).to(device)
buffer_batch2.update_sample_batch(inter_times, mag)

In [30]:
valid_len = (buffer_batch2.end_idx.max().item() + 1)
start_idx= valid_len-2
end_idx = valid_len-1

In [31]:
end_idx

2

In [32]:
buffer_batch2.get_tmp_batch().arrival_times

tensor([[17997.4609],
        [17997.1777],
        [17997.4961],
        [18000.0332],
        [17997.8223],
        [17997.5781],
        [17998.0352],
        [17997.8320],
        [17998.3281],
        [17999.0156],
        [17998.1348],
        [17996.5449],
        [17996.8594],
        [17998.5352],
        [17998.1270],
        [17997.0449],
        [17998.4219],
        [17998.0527],
        [17998.1738],
        [17998.4277],
        [17997.6641],
        [17999.3906],
        [17998.0352],
        [17998.8945],
        [17996.7188],
        [17998.7461],
        [17996.7793],
        [17998.4336],
        [17998.6836],
        [17997.7656],
        [17995.8047],
        [17998.0508],
        [17998.0488],
        [17998.0352],
        [17998.1562],
        [17995.7422],
        [17995.9688],
        [17997.5605],
        [17996.9258],
        [17996.7227],
        [17998.7695],
        [17998.5977],
        [17999.1484],
        [17997.8262],
        [17999.5391],
        [1